# <img src="https://upload.wikimedia.org/wikipedia/commons/6/60/NISAR_artist_concept.jpg" width=400 align="left"/><br><br><br><br>



<img src="https://upload.wikimedia.org/wikipedia/commons/9/9b/NISAR_Mission_Logo.png" width=400 align="left"/><br><br><br><br><br>



# NASA ISRO Synthetic Aperture Radar Mission
## Combined Algorithm Theoretical Basis Document and Jupyter Notebook for <br> *Classification of Wetland Inundation Extent*


Authors: Bruce Chapman, Paul Siqueira

Date: February 15, 2022

Last updated: 
- November 2025, Alexandra Christensen
- December 2024, Brandi Downs


### Summary
This notebook describes the ATBD for generating a wetland inundation product from NISAR time series data stacks. First, the images of the multi-temporal sequence must be well radiometrically calibrated relative to each other, to a higher precision than perhaps required through routine standard calibration of the NISAR imagery. This optional calibration step examines distributed targets that are expected to be unchanged or minimally changed in brightness over a set time span of  an image sequence. With NISAR’s 240 km swath width, it is reasonably assumed that a statistically large area, A<sub>ni</sub>, will not be inundated (or otherwise changing) during any of the 2n observations surrounding the image to be calibrated and classified. These areas will be identified through use of a priori wetlands mask and partly through image segmentation or other methods over the 2n images. <br>
A set of classes will be identified from a multitemporal average of a subset of images including:

- Inundated vegetation (presumption: dominated by double bounce scatter in HH channel) 
- Open water (presumption: low specular scattering in both channels)
- Not inundated (presumption: brighter specular scatter, volume scattering)
- Not classified (presumption: pixels do not align with the scattering model, or no data)

These classes are selected based on calibrated threshold values for the radar backscatter and other metrics. In addition, this same multi-temporal image sequence allows the algorithm to include a more sensitive change detection component for improved robustness. Change detection will allow for refinement within the multitemporal image sequence for change of class during the image sequence that may be more robust than simply classifying the image backscatter and backscatter ratio values.  


### Use environment:
`NISAR_Inundation`

## Step 1: Read in GCOV Data

In [ ]:
import os
import h5py
import numpy as np
import s3fs
import matplotlib.pyplot as plt
import warnings
from yaml import safe_load, safe_dump
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from datetime import datetime
from pathlib import Path
from osgeo import gdal, osr
import rasterio
import rioxarray as rxr
import pandas as pd
from pathlib import Path
import earthaccess
import xarray as xr

gdal.UseExceptions()

### Define papermill parameters

[papermill](https://github.com/nteract/papermill) is a tool that allows you to parameterize and execute a Jupyter Notebook as a script. This notebook has been configured so that it may be executed manually in JupyterLab by running code cells or from the command line, as a script, using papermill.

The code cell below initializes all the variables needed to run the notebook. Do not edit the variables in this code cell.

If running the notebook manually, you will be prompted to input their values as you run the code cells. 

If running from the command line with papermill you can override the variables below with the `-p` argument.

#### papermill command examples:

Run the CalVal data demo from the command line:
```
papermill NISAR_L3_Inundation_ProductGeneration.ipynb NISAR_L3_Inundation_ProductGeneration_papermill_output.ipynb -p papermill True
```

Run on GCOV data for a user-defined AOI and time range:
```
papermill NISAR_L3_Inundation_ProductGeneration.ipynb NISAR_L3_Inundation_ProductGeneration_papermill_output2.ipynb \
-p papermill True -p demo False -p aoi Default \
-p box_left -91.6258 -p box_top 18.7471 -p box_right -92.1666 -p box_bottom 18.403 \
-p start_dt "2025-08-01 00:00:00" -p end_dt "2025-12-01 23:59:59" \
-p granule_name '*_D_*_DHDH_*'
```


In [ ]:
# papermill parameters
##### NEVER EDIT THIS CODE CELL #####
# Either override the variables in this code cell from the command line with papermill or, 
# if running the notebook manually, wait to be prompted for them later in the notebook.

papermill = False
demo = True # downloads calval data
aoi = 'Yucatan_Lake' # Yucatan_Lake, NewOrleans, Default 
box_left, box_top, box_right, box_bottom = None, None, None, None # lat/lon coordinates
start_dt = '2025-08-01 00:00:00'
end_dt = '2025-12-01 23:59:59'
temporal = (start_dt, end_dt)
granule_name = '*_D_*_DHDH_*' # GCOV product ID search filter
indices = '0:4' # If searching for GCOV data, select the index or range of indices for the search results to load

In [ ]:
if not papermill:
    demo_input = input('Run the CalVal demo? [True, False]')
    demo = 'true' == demo_input.lower()
if not demo and not papermill:
    aoi = input('What site? [Yucatan_Lake, NewOrleans, Default]:')
    box_left, box_top, box_right, box_bottom = [input('Define AOI West boundary? [-90.5149]:'),
                                                input('Define AOI North boundary? [18.5015]:'),
                                                input('Define AOI East boundary? [-90.4523]:'),
                                                input('Define AOI South boundary? [18.4451]:')]
    start_dt = input('Start timestamp? [2025-08-01 00:00:00]')
    end_dt = input('End timestamp? [2026-02-01 23:59:59]') 
    temporal = (start_dt, end_dt)
    granule_name = input("GCOV ID search results wildcard filter ['*_D_*_DHDH_*' (filters for decending dual-pol data), *T00888* (filters for a specific CRID)]:")

In [ ]:
main_dir = Path.cwd()
print(main_dir)

In [ ]:
ancillary_dir = main_dir / 'ancillary_data'
aoi_dir = main_dir / aoi 
inun_dir = aoi_dir / 'inundation'
GCOV_dir = aoi_dir / 'GCOV'
TMP_dir = aoi_dir / 'TMP'

Path(ancillary_dir).mkdir(parents = True, exist_ok = True)
Path(aoi_dir).mkdir(parents = True, exist_ok= True)
Path(inun_dir).mkdir(parents = True, exist_ok = True)
Path(GCOV_dir).mkdir(parents = True, exist_ok = True)
Path(TMP_dir).mkdir(parents = True, exist_ok = True)

print(aoi_dir)
print(inun_dir)

In [ ]:
if demo:
    gcov_url = ('s3://nisar-public-ebd/ATBD/ecosystems/inundation/*/*')
    s3 = s3fs.S3FileSystem(anon=True,endpoint_url='https://s3.us-west-1.wasabisys.com')
    all_GCOV_data = ['s3://' + k  for k in s3.glob(gcov_url)]
else:
    kwargs = {
        'granule_name': granule_name,
        'bounding_box': (box_left, box_bottom, box_right, box_top),
        'temporal': temporal,
        'short_name': 'NISAR_L2_GCOV_BETA_V1'
    }
    auth = earthaccess.login()
    s3cred = auth.get_s3_credentials(endpoint="https://nisar.asf.earthdatacloud.nasa.gov/s3credentials")
    results = earthaccess.search_data(**kwargs)
    ## Get URLS for search results
    all_GCOV_data = [earthaccess.results.DataGranule.data_links(x, access ='direct')[0] for x in results if earthaccess.results.DataGranule.data_links(x, access ='direct')[0].endswith('.h5')]
    s3 = s3fs.S3FileSystem(anon = False, key=s3cred['accessKeyId'], 
                           secret = s3cred['secretAccessKey'],
                           token = s3cred['sessionToken'],
                           client_kwargs ={'region_name': 'us-west-2'})
all_GCOV_data = sorted(all_GCOV_data)
print("number of available scenes:", len(all_GCOV_data))

In [ ]:
if not demo and not papermill:
    indices = input('which GCOV files should be used (see Section 1.1 list for numbers)? ex: 0-1, 0, 1, 5, 1:5 inclusive')
    
if ':' in indices:
    indices2 = list(range(int(indices.split(':')[0]), int(indices.split(':')[1])+1))
elif '-' in indices:
    indices2 = list(range(int(indices.split('-')[0]), int(indices.split('-')[1])+1))
elif ',' in indices:
    indices2 = []
    num = indices.split(',')
    for n in range(0,len(num)):
        indices2.append(int(num[n]))
else:
    print('index type not recognized, please rerun the cell')
        

## If you don't want to use all of the images, choose which indices to use now. 
# indices = range(0,19)
print(indices2)
time_series_length = len(indices2)

In [ ]:
import re
from urllib.parse import urlparse
from datetime import datetime
from pathlib import PurePosixPath

NISAR_TS_RE = re.compile(r"_(\d{8}T\d{6})_")

def nisar_start_time_from_url(s3_url: str) -> datetime:
    path = urlparse(s3_url).path
    name = PurePosixPath(path).name
    
    m = NISAR_TS_RE.search(name)
    if not m:
        raise ValueError(f"No NISAR timestamp found in: {s3_url}")
    
    return datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")

datetimes = [nisar_start_time_from_url(all_GCOV_data[i]) for i in indices2]
datetimes

In [ ]:
SAR_images = [pth for i, pth in enumerate(all_GCOV_data) if i in indices2]
SAR_images

In [ ]:
%%time
import xarray as xr
import rioxarray

group_path = "/science/LSAR/GCOV/grids/frequencyA" # change this to any GCOV HDF5 group you wish

kwargs = {
    "cache_type": "background",
    "block_size": 16 * 1024 * 1024,  # 16 MB
}

files = [s3.open(url, "rb", **kwargs) for url in SAR_images]

datatrees = [
    xr.open_datatree(
        f,
        engine="h5netcdf",
        decode_timedelta=False,
        phony_dims="access",
        chunks="auto",
        group=group_path,
    )
    for f in files
]

In [ ]:
%%time
dataarrays = [
    tree.ds.assign_coords(time=dt).expand_dims(time=1) [['HHHH', 'HVHV', 'projection']]
    for dt, tree in zip(datetimes, datatrees)
]

bbox4326 = dict(minx=float(box_left), miny=float(box_bottom), maxx=float(box_right), maxy=float(box_top), crs="EPSG:4326")
for i, da in enumerate(dataarrays):
    da = da.rio.write_crs(f"EPSG:{da.projection.item()}")
    # with rioxarray.set_options(skip_missing_spatial_dims=True):
    dataarrays[i] = da.rio.clip_box(**bbox4326)

ts = xr.concat(dataarrays, dim="time")
ts = ts.persist()
ts

In [ ]:
from collections import Counter

epsgs = [ts.sel(time=t).rio.crs.to_epsg() for t in ts.time]
epsg_equal = len(set(epsgs)) == 1
if epsg_equal:
    print(f"Time series contains single EPSG: {epsgs[0]}")
else:
    raise Exception(f"Time series contains multiple EPSGs: {set(epsgs)}")

num_files = len(ts.time)

In [ ]:
# plot the GCOV images

fig, axs = plt.subplots(num_files, 2, figsize=(12,num_files*5))
cbar_shrink = 0.6

for i, t in enumerate(ts.time):

    im1 = axs[i][0].imshow(ts.sel(time=t).HHHH, vmin=0, vmax=0.5, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, HHHH"
    axs[i][0].set_title(title_str)
    fig.colorbar(im1, ax=axs[i][0], shrink=cbar_shrink)

    im2 = axs[i][1].imshow(ts.sel(time=t).HVHV, vmin=0, vmax=0.1, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, HVHV"
    axs[i][1].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][1], shrink=cbar_shrink)

In [ ]:
%%time

# Compute ratio, product, and sum
ts["hhhh_hvhv_ratio"] = ts.HHHH / ts.HVHV
ts["hhhh_hvhv_product"] = ts.HHHH * ts.HVHV
ts["hhhh_hvhv_sum"] = ts.HHHH + ts.HVHV
ts

In [ ]:
# Plot ratio, product, sum
fig, axs = plt.subplots(num_files, 3, figsize=(14,num_files*5))
cbar_shrink = 0.6

for i, t in enumerate(ts.time):

    im1 = axs[i][0].imshow(ts.sel(time=t).hhhh_hvhv_ratio, vmin=1, vmax=12, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_ratio"
    axs[i][0].set_title(title_str)
    fig.colorbar(im1, ax=axs[i][0], shrink=cbar_shrink)

    im2 = axs[i][1].imshow(ts.sel(time=t).hhhh_hvhv_product, vmin=0, vmax=0.05, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_product"
    axs[i][1].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][1], shrink=cbar_shrink)

    im3 = axs[i][2].imshow(ts.sel(time=t).hhhh_hvhv_sum, vmin=0, vmax=0.5, cmap='gray')
    title_str = f"{str(ts.sel(time=t).time.item().date())}, hhhh_hvhv_sum"
    axs[i][2].set_title(title_str)
    fig.colorbar(im2, ax=axs[i][2], shrink=cbar_shrink)

## Step 2: Read in the classification thresholds

In [ ]:
# Read in contents of yaml configuration file
# with s3.open('nisar-st-data-ondemand/wetlands_config/wetland_calibration_parameters.yaml', 'rb') as f:
with open(main_dir.parent / 'ancillary_data' / 'wetland_calibration_parameters.yaml','rb') as f:    
    config_doc = safe_load(f)

class_thresh = config_doc['runconfig']['calval_sites'][aoi]
class_thresh

From the classification thresholds for this calval site, Yucatan Lake, printed above, we see that only `inun_veg_single_class` and `open_water` are used. For `open_water`, the sum is used, not the product. We will save each threshold next.

In [ ]:
# Save thresholds
comb_method = 'sum'  # combination method is either product or sum; in this case, it's sum
th = {}

# inundated vegetation (iv)
th['iv_hh_max'] = class_thresh['inun_veg_single_class']['HH_max']
th['iv_hh_min'] = class_thresh['inun_veg_single_class']['HH_min']
th['iv_ratio_max'] = class_thresh['inun_veg_single_class']['ratio_max']
th['iv_ratio_min'] = class_thresh['inun_veg_single_class']['ratio_min']

# open water (ow)
if comb_method == 'sum':
    th['ow_comb_max'] = class_thresh['open_water']['sum_max']
    th['ow_comb_min'] = class_thresh['open_water']['sum_min']
elif comb_method == 'product':
    th['ow_comb_max'] = class_thresh['open_water']['product_max']
    th['ow_comb_min'] = class_thresh['open_water']['product_min']    
else:
    raise Exception("Invalid combination method") 

th

## Step 3: Classify the GCOV data

In [ ]:
# First classify open water, then classify remaining (non-open water) pixels as inun veg or not inundated

np.seterr(invalid='ignore')

ts["20m_inundation_classified"] = xr.zeros_like(ts.HHHH)
numpx = ts.isel(time=0).HHHH.notnull().sum().to_numpy()

for t in ts.time:
    ts.sel(time=t)["20m_inundation_classified"] = xr.zeros_like(ts.sel(time=t).HHHH)
    np.zeros(ts.sel(time=t).HHHH.shape, dtype=np.int8)


    # set all valid data pixels to 1
    if comb_method == 'sum':
        ds_comb = ts.sel(time=t).hhhh_hvhv_sum.copy()
    else:
        ds_comb = ts.sel(time=t).hhhh_hvhv_product.copy()
    idx = ds_comb > 0
    ts["20m_inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        1,
        ts.sel(time=t)["20m_inundation_classified"]
    )

    # set open water pixels to 2
    idx = (ds_comb > th['ow_comb_min']) & (ds_comb <= th['ow_comb_max'])
    ts["20m_inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        2,
        ts.sel(time=t)["20m_inundation_classified"]
    )

    # set inundate vegetation pixels to 3
    idx = (ts.sel(time=t).HHHH >= th['iv_hh_min']) & (ts.sel(time=t).HHHH <= th['iv_hh_max']) & \
           (ts.sel(time=t).hhhh_hvhv_ratio >= th['iv_ratio_min']) & (ts.sel(time=t).hhhh_hvhv_ratio <= th['iv_ratio_max']) & \
           (ts.sel(time=t)["20m_inundation_classified"] != 2)
    ts["20m_inundation_classified"].loc[dict(time=t)] = xr.where(
        idx,
        3,
        ts.sel(time=t)["20m_inundation_classified"]
    )

    print(str(ts.time.sel(time=t).item().date()))
    print(f"Open Water: {(100*(ts.sel(time=t)["20m_inundation_classified"] == 2).sum().to_numpy()/numpx):.2f}%")
    print(f"Inundated Vegetation: {(100*(ts.sel(time=t)["20m_inundation_classified"] == 3).sum().to_numpy()/numpx):.2f}%")
    print(f"Not Inundated: {(100*(ts.sel(time=t)["20m_inundation_classified"] == 1).sum().to_numpy()/numpx):.2f}%\n")


In [ ]:
# plot classified images

# set up colormaps
c_white = (255, 255, 255)
c_lightblue = (66, 233, 245)
c_darkblue = (21, 27, 115)
c_gray = (236, 236, 238)

colors = [c_white, c_gray, c_darkblue, c_lightblue]
colors2 = []
for k in colors:
    colors2.append(tuple(np.array(k)/255)) 
cmap = LinearSegmentedColormap.from_list('cmap_class', colors2, N=4)

fig, axs = plt.subplots(num_files, 1, figsize=(6,num_files*5))
cbar_shrink = 0.6
cbar_ticks = [3/8, 9/8, 15/8, 21/8]
cbar_labels = ['no data','not inun','open water','inun veg']
# cbar_ticks = [4/10, 12/10, 20/10, 28/10, 36/10]  # for 2 inun veg classes
# cbar_label = ['no data','not inun','open water','inun veg I','inun veg II']  # for 2 inun veg classes

for i, t in enumerate(ts.time):
    im = axs[i].imshow(ts.sel(time=t)["20m_inundation_classified"], vmin=0, vmax=3, cmap=cmap, interpolation='nearest')
    axs[i].set_title(f"{str(ts.sel(time=t).time.item().date())} 20m_inundation_classified")
    cbar = plt.colorbar(im, ax=axs[i], shrink=cbar_shrink)    
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels(cbar_labels, fontsize=10)  

## Step 4: Aggregate to 1 Hectare

In [ ]:
# output results as geotiff using rasterio

class_dir = Path().cwd() / 'nisar_classifications' / aoi
Path(class_dir).mkdir(parents=True, exist_ok=True)

ts["classification_20m_filepath"] = (
    "time",
    np.full(ts.sizes["time"], "", dtype="U512")
)

ts["classification_1ha_filepath"] = (
    "time",
    np.full(ts.sizes["time"], "", dtype="U512")
)

for i, t in enumerate(ts.time):

    # create 20m geotiffs to use as inputs for gdal Warp
    filepath_20m = str(class_dir / ('nisar_classified_20m_' + datetime.today().strftime('%Y%m%d') + '_gcov_' + str(ts.sel(time=t).time.item().date()) + \
                      '_' + aoi + '.tif'))
    ts["classification_20m_filepath"].loc[{"time": t}] = filepath_20m
    meta = {'driver': 'GTiff', 
            'dtype': 'float32', 
            'nodata': None, 
            'width': ts.sizes["xCoordinates"], 
            'height': ts.sizes["yCoordinates"], 
            'count': 1, 
            'crs': ts.rio.crs, 
            'transform': ts.rio.transform(),
            'tiled': False, 
            'interleave': 'band'}
    with rasterio.open(filepath_20m, 'w', **meta) as dst:
        dst.write(ts.sel(time=t)["20m_inundation_classified"], indexes=1)    

    L3_filepath_1ha = str(class_dir / ('nisar_classified_1ha_' + datetime.today().strftime('%Y%m%d') + '_gcov_' + str(ts.sel(time=t).time.item().date()) + \
                      '_' + aoi + '.tif'))
    ts["classification_1ha_filepath"].loc[{"time": t}] = L3_filepath_1ha
    gdal.Warp(L3_filepath_1ha, filepath_20m, xRes=100, yRes=-100, resampleAlg=gdal.GRA_Mode, format="COG")

    # optional: remove 20m files
    Path(filepath_20m).unlink()

In [ ]:
# read in the 1 ha tif data

nisar_classified_1ha = []

for t in ts.time:
    ds = rxr.open_rasterio(ts.sel(time=t)["classification_1ha_filepath"].item())
    nisar_classified_1ha.append(ds.to_numpy().squeeze().astype(np.int8))

In [ ]:
# plot 1-ha classified images

# set up colormaps
c_white = (255, 255, 255)
c_lightblue = (66, 233, 245)
c_darkblue = (21, 27, 115)
c_gray = (236, 236, 238)

colors = [c_white, c_gray, c_darkblue, c_lightblue]
colors2 = []
for k in colors:
    colors2.append(tuple(np.array(k)/255)) 
cmap = LinearSegmentedColormap.from_list('cmap_class', colors2, N=4)

fig, axs = plt.subplots(num_files, 1, figsize=(6,num_files*5))
cbar_shrink = 0.6
cbar_ticks = [3/8, 9/8, 15/8, 21/8]
cbar_labels = ['no data','not inun','open water','inun veg']
# cbar_ticks = [4/10, 12/10, 20/10, 28/10, 36/10]  # for 2 inun veg classes
# cbar_label = ['no data','not inun','open water','inun veg I','inun veg II']  # for 2 inun veg classes

for i, t in enumerate(ts.time):
    im = axs[i].imshow(nisar_classified_1ha[i], vmin=0, vmax=3, cmap=cmap, interpolation='nearest')
    axs[i].set_title(f"{str(ts.sel(time=t).time.item().date())} 1ha_inundation_classified")
    cbar = plt.colorbar(im, ax=axs[i], shrink=cbar_shrink)    
    cbar.set_ticks(cbar_ticks)
    cbar.set_ticklabels(cbar_labels, fontsize=10)    